In [1]:
import pyarrow.parquet as pq
import json
import glob
import random

files = glob.glob("data/*.parquet")
random.shuffle(files)

TARGET_ARTICLES = 1500000
count = 0

output_path = "wiki.txt"


def article_to_text(article):
    title = article.get("name", "")
    abstract = article.get("abstract", "")

    # filtr disambiguation pages
    if abstract and "may refer to" in abstract.lower():
        return None

    sections_raw = article.get("sections", "[]")

    try:
        sections = json.loads(sections_raw)
    except:
        return None

    text_parts = []

    text_parts.append(f"TITLE: {title}\n")

    if abstract and abstract.strip():
        text_parts.append("\nABSTRACT:\n" + abstract.strip() + "\n")

    has_content = False

    for sec in sections:
        sec_name = sec.get("name", "").strip()
        if not sec_name:
            continue

        section_text = []

        for part in sec.get("has_parts", []):
            if part.get("type") == "paragraph":
                val = part.get("value", "").strip()
                if val:
                    section_text.append(val)

        # delete empty sections
        if not section_text:
            continue

        has_content = True

        text_parts.append(f"\n## {sec_name}\n")
        text_parts.append("\n".join(section_text) + "\n")

    if not has_content:
        return None

    text_parts.append("\n<|endoftext|>\n")

    return "".join(text_parts)


with open(output_path, "w", encoding="utf-8") as out:

    for file in files:
        if count >= TARGET_ARTICLES:
            break

        print("Processing:", file)

        pf = pq.ParquetFile(file)

        # shuffle row groups
        row_groups = list(range(pf.num_row_groups))
        random.shuffle(row_groups)

        for rg in row_groups:
            table = pf.read_row_group(rg)
            rows = table.to_pylist()

            random.shuffle(rows)

            for article in rows:
                if count >= TARGET_ARTICLES:
                    break

                txt = article_to_text(article)

                if txt:
                    out.write(txt + "\n")
                    count += 1
        print(f"{count} / {TARGET_ARTICLES}")

print(f"DONE -> wiki.txt ({count} articles)")

Processing: data/enwiki_namespace_0_00175.parquet
24770 / 1500000
Processing: data/enwiki_namespace_0_00112.parquet
45999 / 1500000
Processing: data/enwiki_namespace_0_00196.parquet
70776 / 1500000
Processing: data/enwiki_namespace_0_00032.parquet
95077 / 1500000
Processing: data/enwiki_namespace_0_00118.parquet
116791 / 1500000
Processing: data/enwiki_namespace_0_00065.parquet
137903 / 1500000
Processing: data/enwiki_namespace_0_00145.parquet
162492 / 1500000
Processing: data/enwiki_namespace_0_00240.parquet
196120 / 1500000
Processing: data/enwiki_namespace_0_00023.parquet
220267 / 1500000
Processing: data/enwiki_namespace_0_00227.parquet
247124 / 1500000
Processing: data/enwiki_namespace_0_00107.parquet
271702 / 1500000
Processing: data/enwiki_namespace_0_00086.parquet
295955 / 1500000
Processing: data/enwiki_namespace_0_00181.parquet
320718 / 1500000
Processing: data/enwiki_namespace_0_00090.parquet
345159 / 1500000
Processing: data/enwiki_namespace_0_00174.parquet
378548 / 1500000

In [ ]:
import pyarrow.parquet as pq
import json
import glob
import random

def load_titles(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return {line.strip() for line in f if line.strip()}
    except FileNotFoundError:
        print(f"File {filepath} not found – treated as empty.")
        return set()

titles2_set = load_titles("titles2.txt")
notitles_set = load_titles("notitles.txt")

files = glob.glob("data/*.parquet")
TARGET_ARTICLES = 1_500_000
output_path = "wiki.txt"

count = 0
added_titles = set()

def article_to_text(article):
    title = article.get("name", "")
    abstract = article.get("abstract", "")

    if abstract and "may refer to" in abstract.lower():
        return None

    sections_raw = article.get("sections", "[]")
    try:
        sections = json.loads(sections_raw)
    except:
        return None

    text_parts = []
    text_parts.append(f"TITLE: {title}\n")

    if abstract and abstract.strip():
        text_parts.append("\nABSTRACT:\n" + abstract.strip() + "\n")

    has_content = False

    for sec in sections:
        sec_name = sec.get("name", "").strip()
        if not sec_name:
            continue

        section_text = []
        for part in sec.get("has_parts", []):
            if part.get("type") == "paragraph":
                val = part.get("value", "").strip()
                if val:
                    section_text.append(val)

        if not section_text:
            continue

        has_content = True
        text_parts.append(f"\n## {sec_name}\n")
        text_parts.append("\n".join(section_text) + "\n")

    if not has_content:
        return None

    text_parts.append("\n<|endoftext|>\n")
    return "".join(text_parts)

with open(output_path, "w", encoding="utf-8") as out:

    print("Phase 1 – collecting articles from titles2.txt (including blacklisted titles)...")
    for file in files:
        if count >= TARGET_ARTICLES:
            break
        print("Processing (titles2):", file)

        pf = pq.ParquetFile(file)
        for rg in range(pf.num_row_groups):
            if count >= TARGET_ARTICLES:
                break
            table = pf.read_row_group(rg)
            rows = table.to_pylist()
            for article in rows:
                if count >= TARGET_ARTICLES:
                    break
                title = article.get("name", "")
                if title in titles2_set and title not in added_titles:
                    txt = article_to_text(article)
                    if txt:
                        out.write(txt + "\n")
                        count += 1
                        added_titles.add(title)
                        titles2_set.discard(title)
        print(f"Progress after this file: {count} / {TARGET_ARTICLES}")

    if count < TARGET_ARTICLES:
        print("Phase 2 – filling up with random articles (excluding blacklisted titles)...")
        random.shuffle(files)

        for file in files:
            if count >= TARGET_ARTICLES:
                break
            print("Processing (fill):", file)

            pf = pq.ParquetFile(file)
            row_groups = list(range(pf.num_row_groups))
            random.shuffle(row_groups)

            for rg in row_groups:
                if count >= TARGET_ARTICLES:
                    break
                table = pf.read_row_group(rg)
                rows = table.to_pylist()
                random.shuffle(rows)

                for article in rows:
                    if count >= TARGET_ARTICLES:
                        break
                    title = article.get("name", "")
                    if title in added_titles or title in notitles_set:
                        continue

                    txt = article_to_text(article)
                    if txt:
                        out.write(txt + "\n")
                        count += 1
                        added_titles.add(title)
            print(f"Progress after this file: {count} / {TARGET_ARTICLES}")

print(f"DONE -> {output_path} ({count} articles)")

Phase 1 – collecting articles from titles2.txt (including blacklisted titles)...
Processing (titles2): data/enwiki_namespace_0_00134.parquet
Progress after this file: 3530 / 1500000
Processing (titles2): data/enwiki_namespace_0_00007.parquet
Progress after this file: 4827 / 1500000
Processing (titles2): data/enwiki_namespace_0_00077.parquet
Progress after this file: 6834 / 1500000
Processing (titles2): data/enwiki_namespace_0_00226.parquet
Progress after this file: 9410 / 1500000
Processing (titles2): data/enwiki_namespace_0_00071.parquet
Progress after this file: 11050 / 1500000
Processing (titles2): data/enwiki_namespace_0_00218.parquet
Progress after this file: 12829 / 1500000
Processing (titles2): data/enwiki_namespace_0_00006.parquet
Progress after this file: 13933 / 1500000
Processing (titles2): data/enwiki_namespace_0_00083.parquet
Progress after this file: 15832 / 1500000
Processing (titles2): data/enwiki_namespace_0_00087.parquet
Progress after this file: 17900 / 1500000
Proce

In [6]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /home/mzums/programming/fakewiki/dev


In [7]:
import os
size = os.path.getsize("wiki.txt")
print(f"File size: {size / (1024**3):.2f} GB")

File size: 4.08 GB


In [ ]:
import unicodedata
import re

def clean_text(text):
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    return text

input_file = "wiki.txt"
output_file = "wiki_clean.txt"

with open(input_file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="ascii", errors="ignore") as fout:
    
    for line in fin:
        cleaned_line = clean_text(line)
        fout.write(cleaned_line)